In [ ]:
# run this notebook after 18_annotate_trio_sibling_dnm_COSMIC_python
# use this notebook to prepare files for plotting the distribution of sibling DNMs across the genome
# We need UKB sibling DNMs (anonymous) of specific mutation types 
#(./ukb/ukb_sibs_dnms_cpg_tpg.csv, ./ukb/ukb_sibs_dnms_c_a.csv, ./ukb/ukb_sibs_dnms_c_g_chr8.csv)
# after this, run 20_distribution_genome_figure_files_python

In [ ]:
library(jcolors)
library(data.table)
library(ggplot2)
library(dplyr)
library(jsonlite)
library(tidyverse)
library(ggpubr)
library(purrr)
library(tidyr)
library(emmeans)
library(RColorBrewer)
library(GenomicRanges)
library(rtracklayer)
library(grid)
library(cowplot)
library(bedtoolsr)

In [ ]:

# ── 1. Create 100kb windows across chr8:1-7Mb ────────────────────────────────
windows <- tibble(
  chrom   = "chr8",
  start = seq(1e6, 30e6 - 1e6, by = 1e6),
  end   = seq(2e6, 30e6, by = 1e6)
)

giab_bed = fread("./data/giab_difficult_merged_oct27.bed")
giab_bed = data.frame(chrom = paste0("chr", giab_bed$chrom), start=giab_bed$start,
                     end = giab_bed$end)
windows$num_bases_giab = NA
for (i in 1:nrow(windows)){
    sub = data.frame(chrom = windows$chrom[i], start = windows$start[i], end = windows$end[i])
    merged = bedtoolsr::bt.intersect(sub, giab_bed)
    windows$num_bases_giab[i] = sum(merged$V3-merged$V2+1)
}
aou_difs = fread("./sibs_snps_QC_GIAB_distance_AF_outliers_COSMIC.txt")
aou_cpg = aou_difs %>% filter(substr(Type, 3,7) == "C>T]G") %>% select(chr, pos)
aou_ca = aou_difs %>% filter(substr(Type, 3,5) == "C>A") %>% select(chr, pos)
aou_cg = aou_difs %>% filter(substr(Type, 3,5) == "C>G") %>% select(chr, pos) %>% filter(chr == "8")

ukb_cpg = fread("./ukb/ukb_sibs_dnms_cpg_tpg.csv", sep = ",")
ukb_ca = fread("./ukb/ukb_sibs_dnms_c_a.csv", sep = ",")
ukb_cg = fread("./ukb/ukb_sibs_dnms_c_g_chr8.csv", sep = ",")
names(ukb_cpg) = c('chr', 'pos')
names(ukb_ca) = c('chr', 'pos')
names(ukb_cg) = c('chr', 'pos')
aou_cpg$dataset = 'aou'
ukb_cpg$dataset = 'ukb'
aou_ca$dataset = 'aou'
ukb_ca$dataset = 'ukb'
aou_cg$dataset = 'aou'
ukb_cg$dataset = 'ukb'
cpg = rbind(aou_cpg, ukb_cpg)
cpg$chr = paste0("chr", cpg$chr)
ca = rbind(aou_ca, ukb_ca)
ca$chr = paste0("chr", ca$chr)
# count mutations per cg 
windows$n_aou_cg = NA
windows$n_aou_ca = NA
windows$n_aou = NA
windows$n_ukb_cg = NA
windows$n_ukb_ca = NA
windows$n_ukb = NA
aou_difs_else = aou_difs %>% filter(substr(Type, 3,5) == "C>G")
aou_bed = data.frame(chrom = paste0("chr", aou_difs$chr),
                    start = aou_difs$pos, end = aou_difs$pos+1)
aou_cg_bed = data.frame(chrom = paste0("chr", aou_cg$chr),
                   start = aou_cg$pos, end = aou_cg$pos + 1)
ukb_bed = data.frame(chrom = paste0("chr", ukb_difs$chr),
                    start = ukb_difs$pos, end = ukb_difs$pos+1)
ukb_cg_bed = data.frame(chrom = paste0("chr", ukb_cg$chr),
                   start = ukb_cg$pos, end = ukb_cg$pos + 1)
ukb_ca_bed = data.frame(chrom = paste0("chr", ukb_ca$chr),
                   start = ukb_ca$pos, end = ukb_ca$pos + 1)
aou_ca_bed = data.frame(chrom = paste0("chr", aou_ca$chr),
                   start = aou_ca$pos, end = aou_ca$pos + 1)

for (i in 1:nrow(windows)){
    sub = data.frame(chrom = windows$chrom[i], start = windows$start[i], end = windows$end[i])
    aou_merged = bedtoolsr::bt.intersect(sub, aou_bed, c = TRUE)
    aou_cg_merged = bedtoolsr::bt.intersect(sub, aou_cg_bed , c = TRUE)
    aou_ca_merged = bedtoolsr::bt.intersect(sub, aou_ca_bed , c = TRUE)
    ukb_merged = bedtoolsr::bt.intersect(sub, ukb_bed, c = TRUE)
    ukb_cg_merged = bedtoolsr::bt.intersect(sub, ukb_cg_bed , c = TRUE)
    ukb_ca_merged = bedtoolsr::bt.intersect(sub, ukb_ca_bed , c = TRUE)
    windows$n_aou_cg[i] = aou_cg_merged$V4
    windows$n_aou_ca[i] = aou_ca_merged$V4
    windows$n_aou[i] = aou_merged$V4
    windows$n_ukb_cg[i] = ukb_cg_merged$V4
    windows$n_ukb[i] = ukb_merged$V4
    windows$n_ukb_ca[i] = ukb_ca_merged$V4

}
fwrite(windows, "./chr8_cg.tsv", sep = "\t", row.names = FALSE, quote = FALSE)

In [ ]:
aou_difs = fread("./sibs_snps_QC_GIAB_distance_AF_outliers_COSMIC.txt")
ukb_difs = fread("./ukb/ukb_sibs_dnms_all.csv", sep = ",")
giab = fread("./data/giab_difficult_merged_oct27.bed")
names(giab) = c("chr", "start", "end", "length")
aou_difs = aou_difs %>% select("chr", "pos")
aou_difs$dataset = "aou"
names(ukb_difs) = c("chr", "pos")
ukb_difs$dataset = "ukb"
dnms = rbind(aou_difs, ukb_difs)
dnms$chr = paste0("chr", dnms$chr)
giab$chr = paste0("chr", giab$chr)
fwrite(dnms, "./all_difs.tsv", sep = "\t", 
      row.names = FALSE, quote = FALSE)

In [ ]:
fwrite(cpg, "./cpg_mutations.tsv", sep = "\t", row.names = FALSE, quote = FALSE)